# Exemplo 1: Lazy Evaluation para funções genéricas

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
import dask
import time

# Funções que simulam um processo demorado
def demorar_para_somar(x, y):
    time.sleep(1)
    return x + y

def demorar_para_multiplicar(x, y):
    time.sleep(1)
    return x * y

# 1. Transformar as funções em "delayed" (preguiçosas)
# Você pode usar a função dask.delayed() ou o decorator @dask.delayed
soma_preguicosa = dask.delayed(demorar_para_somar)
multiplicacao_preguicosa = dask.delayed(demorar_para_multiplicar)

# 2. Construir o grafo de tarefas
# Nenhuma função é executada imediatamente aqui
resultado_soma1 = soma_preguicosa(1, 2)
resultado_soma2 = soma_preguicosa(3, 4)

resultado_final = multiplicacao_preguicosa(resultado_soma1, resultado_soma2)

# 3. Executar o grafo em paralelo ao chamar compute()
inicio = time.time()
saida = resultado_final.compute()
fim = time.time()

print(f"Resultado final: {saida}")
print(f"Tempo total de execução: {fim - inicio:.2f} segundos")
print()

### MESMAS OPERAÇÕES SEM LAZY EVALUATION
inicio = time.time()
result_sum1 = demorar_para_somar(1, 2)
result_sum2 = demorar_para_somar(3, 4)
result_mul = demorar_para_multiplicar(result_sum1, result_sum2)
fim = time.time()

print(f"Resultado final sem lazy eval: {saida}")
print(f"Tempo total de execução sem lazy eval: {fim - inicio:.2f} segundos")


Resultado final: 21
Tempo total de execução: 2.00 segundos

Resultado final sem lazy eval: 21
Tempo total de execução sem lazy eval: 3.00 segundos


### Segunda forma de escrever o exemplo 1

In [2]:
import dask
from dask import delayed
import time

@delayed
def demorar_para_somar(x, y):
    time.sleep(1)
    return x + y

@delayed
def demorar_para_multiplicar(x, y):
    time.sleep(1)
    return x * y


# 1. Construir o grafo de tarefas
resultado_soma1 = demorar_para_somar(1, 2)
resultado_soma2 = demorar_para_somar(3, 4)

resultado_final = demorar_para_multiplicar(resultado_soma1, resultado_soma2)

# 2. Executar o grafo em paralelo ao chamar compute()
inicio = time.time()
saida = resultado_final.compute()
fim = time.time()

print(f"Resultado final: {saida}")
print(f"Tempo total de execução: {fim - inicio:.2f} segundos")
print()



Resultado final: 21
Tempo total de execução: 2.00 segundos



# Exemplo 2: Paralelismo para GridSearchCV

In [2]:
import time
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
import joblib
from dask.distributed import Client

# 1. Iniciar o Client do Dask localmente
client = Client(dashboard_address=":0")  # porta livre escolhida pelo SO
print(f"Abra o Dashboard do Dask clicando aqui: {client.dashboard_link}\n")

# 2. Criar um dataset sintético razoavelmente pesado (7500 linhas, 20 colunas)
print("Gerando dataset sintético...")
X, y = make_classification(n_samples=7500, n_features=20, random_state=42)

# 3. Configurar o modelo (Support Vector Machine) e os hiperparâmetros
modelo = SVC()
param_grid = {
    'C':      [0.1, 1, 10, 100],
    'gamma':  [0.1, 0.01, 0.001, 0.0001],
    'kernel': ['rbf', 'poly', 'sigmoid']
}

# cv=3 fará validação cruzada 3 vezes.
# Total de modelos treinados: 4 * 4 * 3 * 3 (cv) = 144 treinamentos
grid_search = GridSearchCV(modelo, param_grid, cv=3, n_jobs=-1)

print("Iniciando o GridSearch. Vá olhar o Dashboard agora!")
inicio = time.time()

# O context manager do joblib faz o Scikit-Learn enviar o trabalho para o Dask
with joblib.parallel_backend('dask'):
    grid_search.fit(X, y)

fim = time.time()

print("\n--- Treinamento Concluído! ---")
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Tempo total de execução: {fim - inicio:.2f} segundos")

client.close()

/pesquisa-dask/.venv/lib/python3.12/site-packages/rapids_dask_dependency/dask_loader.py:36: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  return importlib.import_module(spec.name)
/pesquisa-dask/.venv/lib/python3.12/site-packages/rapids_dask_dependency/dask_loader.py:36: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  return importlib.import_module(spec.name)
/pesquisa-dask/.venv/lib/python3.12/site-packages/rapids_dask_dependency/dask_loader.py:36: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  retu

Abra o Dashboard do Dask clicando aqui: http://127.0.0.1:40695/status

Gerando dataset sintético...
Iniciando o GridSearch. Vá olhar o Dashboard agora!

--- Treinamento Concluído! ---
Melhores parâmetros encontrados: {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
Tempo total de execução: 86.55 segundos


In [ ]:
import time
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# 1. Criar um dataset sintético razoavelmente pesado (10 mil linhas, 20 colunas)
print("Gerando dataset sintético...")
X, y = make_classification(n_samples=7500, n_features=20, random_state=42)

# 2. Configurar o modelo (Support Vector Machine) e os hiperparâmetros
modelo = SVC()
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.1, 0.01, 0.001, 0.0001],
    'kernel': ['rbf', 'poly', 'sigmoid']
}

# n_jobs=-1 instrui o scikit-learn a executar as tarefas em paralelo
# cv=3 fará validação cruzada 3 vezes. 
# Total de modelos treinados: 4 * 4 * 3 * 3 (cv) = 144 treinamentos
grid_search = GridSearchCV(modelo, param_grid, cv=3, n_jobs=-1)

inicio = time.time()

grid_search.fit(X, y)

fim = time.time()

print("\n--- Treinamento Concluído! ---")
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Tempo total de execução: {fim - inicio:.2f} segundos")

Gerando dataset sintético...

--- Treinamento Concluído! ---
Melhores parâmetros encontrados: {'C': 100, 'gamma': 0.001, 'kernel': 'rbf'}
Tempo total de execução: 79.47 segundos


# Exemplo 3: Dados > RAM

In [3]:
import dask
import dask.dataframe as dd
import time

# 1. Leitura LAZY — nenhum dado entra na RAM aqui
ddf = dd.read_csv("data/dados_*.csv")

print(f"Partições : {ddf.npartitions}  (uma por arquivo, lidas sob demanda)")
print(f"Colunas   : {list(ddf.columns)}\n")

# 2. Operações out-of-core — Dask processa partição por partição
#    A RAM nunca armazena os 10 M de linhas ao mesmo tempo
media_por_cat = ddf.groupby("categoria")["valor"].mean()
soma_por_cat  = ddf.groupby("categoria")["quantidade"].sum()
n_total       = ddf["id"].count()

# 3. compute() dispara a execução de todos os grafos de uma vez
inicio = time.time()
medias, somas, total = dask.compute(media_por_cat, soma_por_cat, n_total, scheduler='synchronous')
fim = time.time()

print(f"Registros processados : {total:,}")
print(f"\nMédia de 'valor'      :\n{medias.round(2)}")
print(f"\nSoma de 'quantidade'  :\n{somas}")
print(f"\nTempo de execução     : {fim - inicio:.2f}s")


Partições : 10  (uma por arquivo, lidas sob demanda)
Colunas   : ['id', 'categoria', 'valor', 'quantidade']

Registros processados : 10,000,000

Média de 'valor'      :
categoria
A   -0.04
B   -0.09
C    0.04
D    0.03
Name: valor, dtype: float64

Soma de 'quantidade'  :
categoria
A    624709820
B    625413034
C    625386333
D    624848455
Name: quantidade, dtype: int64

Tempo de execução     : 3.33s


# Exemplo 4: Integração com Pytorch


In [4]:
## APENAS DEFINIÇÃO DO MODELO E FUNÇÃO QUE SERÁ UTILIZADA
import dask.dataframe as dd
import pandas as pd
import torch
import torch.nn as nn
import time

# ── Modelo PyTorch ────────────────────────────────────────────────────────
class Classificador(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(
            nn.Linear(2, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),  nn.Sigmoid(),
        )
    def forward(self, x):
        return self.rede(x)

# Pesos são carregados uma vez e reutilizados em cada partição
pesos = Classificador().state_dict()

def inferir_lote(df: pd.DataFrame) -> pd.Series:
    modelo = Classificador()
    modelo.load_state_dict(pesos)
    modelo.eval()
    X = torch.tensor(df[["valor", "quantidade"]].values, dtype=torch.float32)
    with torch.no_grad():
        scores = modelo(X).numpy().flatten()
    return pd.Series(scores, index=df.index, name="score")



In [5]:
# ── Leitura lazy dos 10 M de linhas gerados no Exemplo 3 ─────────────────
# Com pandas: pd.read_csv() carregaria tudo na RAM de uma vez (~1 GB)
# Com Dask: lê e descarta uma partição por vez — RAM constante
ddf = dd.read_csv("data/dados_*.csv")
print(f"Partições : {ddf.npartitions}  |  Colunas : {list(ddf.columns)}\n")

# ── Inferência out-of-core ────────────────────────────────────────────────
resultado = ddf.map_partitions(
    inferir_lote,
    meta=pd.Series(dtype="float32", name="score"),
)

print("Rodando inferência PyTorch em 10 M de linhas (sem Client distribuído)...")
inicio = time.time()
scores = resultado.compute()
fim = time.time()

print(f"\nLinhas processadas : {len(scores):,}")
print(f"Primeiros scores   : {scores.head().values.round(4)}")
print(f"Tempo total        : {fim - inicio:.2f}s")
print("\nA RAM nunca armazenou os 10 M de linhas ao mesmo tempo!")


Partições : 10  |  Colunas : ['id', 'categoria', 'valor', 'quantidade']

Rodando inferência PyTorch em 10 M de linhas (sem Client distribuído)...

Linhas processadas : 10,000,000
Primeiros scores   : [1. 1. 1. 1. 1.]
Tempo total        : 3.23s

A RAM nunca armazenou os 10 M de linhas ao mesmo tempo!


In [ ]:
# SE TENTARMOS COM PANDAS O KERNEL MORRE
import glob
import pandas as pd
arquivos = sorted(glob.glob("data/dados_*.csv"))
df = pd.concat([pd.read_csv(f) for f in arquivos])
print(f"Partições : {df.shape[0]}  |  Colunas : {list(df.columns)}\n")

print("Rodando inferência PyTorch em 10 M de linhas...")
inicio = time.time()
resultado = inferir_lote(df)
fim = time.time()

print(f"\nLinhas processadas : {len(scores):,}")
print(f"Primeiros scores   : {scores.head().values.round(4)}")
print(f"Tempo total        : {fim - inicio:.2f}s")

Partições : 10000000  |  Colunas : ['id', 'categoria', 'valor', 'quantidade']

Rodando inferência PyTorch em 10 M de linhas...

Linhas processadas : 10,000,000
Primeiros scores   : [1. 1. 1. 1. 1.]
Tempo total        : 6.84s

A RAM nunca armazenou os 10 M de linhas ao mesmo tempo!
